In [1]:
import pandas as pd
import pickle
import sys
import logging

### clean the pub file with new column names, define data type and output the mapping between ost_bk and uid

In [2]:
pub_path = "../../data/dropbox/LaborDivision/data/rawdata/article_labordivision.txt"

In [3]:
pub_df = pd.read_csv(pub_path,sep="|",on_bad_lines='skip')

In [4]:
pub_df.head()

,UID,OST_BK,Annee_Bibliographique,Code_Discipline,Titre
0,WOS:000419403800078,69470021,2018,18,From early stress to 12-month development in v...
1,WOS:000513294200019,69470022,2020,77,Evolution of robust high speed optical-flow-ba...
2,WOS:000362235300024,69470023,2015,41,Activation of the GLP-1 Receptor by Liraglutid...
3,WOS:000326278200002,69470024,2013,58,A 7-Year Case of Furosemide-Induced Immune Thr...
4,WOS:000305602900042,69470046,2012,77,Study of semiconducting nanomaterials under pr...


In [5]:
pub_df.columns=['uid','ost_bk','year','dis','title']
pub_df['ost_bk'] = pub_df['ost_bk'].astype(str)
pub_df['year'] = pub_df['year'].astype(str)
pub_df['dis'] = pub_df['dis'].astype(str)

In [28]:
ostbk2uid = dict(zip(pub_df.ost_bk,pub_df.uid))
with open("../../data/dropbox/LaborDivision/Data/cleanedraw/ostbk_uid.pkl",'wb') as pickle_file:
    pickle.dump(ostbk2uid, pickle_file)

In [26]:
pub_df[['ost_bk','year','dis','title']].to_pickle("../../data/dropbox/LaborDivision/Data/cleanedraw/pub.pkl")

In [49]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def return_pub_no_large_team(df):
    '''return articles that team size smaller than n'''
    ids = df[df.isna().any(axis=1)].OST_BK.unique()
    logger.info('{} of papers NaN values'.format(len(ids)))
    
    return df[~df.OST_BK.isin(ids)]


In [12]:

def return_pub_with_first(df, total):
    '''return articles with first authors'''
    withfirst = df[df.Seq_No==1].OST_BK.unique()
    logger.info('{} of papers have first author assigned'.format(len(withfirst)/total))
    
    return df[df.OST_BK.isin(withfirst)]

def return_pub_with_author(pub_df, author_df, total):
    merged_df = pub_df.merge(author_df, on=['OST_BK','Seq_No'], how='left')
    invalid_ids = merged_df[merged_df.cluster_id.isna()].OST_BK.unique()
    valid_df = merged_df[~merged_df.OST_BK.isin(invalid_ids)]
    logger.info('{} of papers have incomplete author information'.format(len(invalid_ids)/total))
    
    return valid_df

def remove_multi_affiliation(df):
    new_df = df.sort_values(by=['OST_BK','Seq_No','Addr_No']
                           ).drop_duplicates(subset=['OST_BK','Seq_No'],keep='first')
    
    return new_df

def replace_cntry_name_dropna(df, name2code):
    df['cntry'] = df['Country'].map(name2code)
    invalid_ids = df[df.cntry.isna()].OST_BK.unique()
    logger.info('countries without iso-3 code: {}'.format(df[df.cntry.isna()].Country.unique()))
    
    return df[~df.OST_BK.isin(invalid_ids)]



In [13]:

cntry_path = '../../data/dropbox/LaborDivision/data/rawdata/article_cntry.txt'
author_path ='../../data/dropbox/LaborDivision/data/rawdata/article_author_info.txt'
name2code_path = '../../data/dropbox/LaborDivision/data/additionaldata/cntry_meta.csv'



cntry_raw = pd.read_csv(cntry_path, sep="|", on_bad_lines='skip', dtype={'OST_BK':'int64','Seq_No':'float64'})
author_info = pd.read_csv(author_path, sep=",", on_bad_lines='skip', dtype={'OST_BK':'int64','Seq_No':'float64'})
name2code_df = pd.read_csv(name2code_path).dropna()
name2code_dict = dict(zip(name2code_df.cntry,name2code_df['Code']))

pub_total_raw = len(cntry_raw.OST_BK.unique())
logger.info('{} number of pubs in raw data'.format(pub_total_raw))


/tmp/ipykernel_1284663/337769274.py:8: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  author_info = pd.read_csv(author_path, sep=",", on_bad_lines='skip', dtype={'OST_BK':'int64','Seq_No':'float64'})
INFO:__main__:20211823 number of pubs in raw data


In [50]:
return_pub_no_large_team(cntry_raw)

INFO:__main__:1168748 of papers NaN values


,OST_BK,Seq_No,Last_Name,Addr_No,Country
0,102266829,1.0,Lallukka,1.0,Finland
1,80146396,8.0,Evans,2.0,Iraq
2,83186211,2.0,Epping,2.0,Canada
3,59347767,23.0,Ramos-Casals,1.0,Spain
4,100396018,6.0,Volodarsky-Perel,1.0,Canada
...,...,...,...,...,...
127029266,37680509,4.0,Saba,1.0,USA
127029267,40215283,8.0,Ma,1.0,Canada
127029268,88146584,2.0,Hill,1.0,USA
127029269,82110841,13.0,Mizowaki,1.0,Japan


In [51]:
1168748/20211823

0.057824967099701995

In [23]:
miss = set(cntry_raw.OST_BK.unique())-set(cntry_df.OST_BK.unique())

In [ ]:


cntry_df = return_pub_with_first(cntry_df, pub_total_raw)

cntry_df = return_pub_with_author(cntry_df, author_info, pub_total_raw)

cntry_df = remove_multi_affiliation(cntry_df)

cntry_df = replace_cntry_name_dropna(cntry_df, name2code_dict)

cntry_df.rename(columns={'OST_BK':'ost_bk','Seq_No':'seqno', 'Last_Name':'lastname'}, inplace=True)
cntry_df.loc[:,'seqno'] = cntry_df['seqno'].astype('int64')
cntry_df.loc[:,'cluster_id'] = cntry_df['cluster_id'].astype('int64')


In [20]:
   
logger.info('{}({}) papers are kept in the analysis'.format(len(cntry_df.ost_bk.unique()),len(cntry_df.ost_bk.unique())/pub_total_raw))

cntry_df[['ost_bk','seqno','cluster_id','lastname','cntry']].head().to_pickle("test.pkl")


INFO:__main__:17380209(0.8599030874157171) papers are kept in the analysis


In [3]:
cntry_path = '../../data/dropbox/LaborDivision/data/rawdata/article_cntry.txt'
cntry_raw = pd.read_csv(cntry_path, sep="|", on_bad_lines='skip')
cntry_raw.head()

,OST_BK,Seq_No,Last_Name,Addr_No,Country
0,102266829,1.0,Lallukka,1.0,Finland
1,80146396,8.0,Evans,2.0,Iraq
2,83186211,2.0,Epping,2.0,Canada
3,59347767,23.0,Ramos-Casals,1.0,Spain
4,100396018,6.0,Volodarsky-Perel,1.0,Canada


In [17]:
test.loc[:,'Seq_No'] = test['Seq_No'].astype('int64')

In [16]:
test.dtypes

OST_BK         int64
Seq_No         int64
Last_Name     object
Addr_No      float64
Country       object
dtype: object

In [14]:
cntry_raw.dtypes

OST_BK         int64
Seq_No       float64
Last_Name     object
Addr_No      float64
Country       object
dtype: object

In [7]:
len(cntry_raw.OST_BK.unique())

20211823

In [8]:
1154714/20211823

0.057130621023150654

In [8]:
def return_pub_no_large_team(df, n):
    '''return articles that team size smaller than n'''
    ids = df[df.Seq_No>n].OST_BK.unique()
    return df[~df.OST_BK.isin(ids)]

In [9]:
cntry_df = return_pub_no_large_team(cntry_raw,10)

In [10]:
def return_pub_with_first(df):
    '''return articles with first authors'''
    withfirst = df[df.Seq_No==1].OST_BK.unique()
    return df[df.OST_BK.isin(withfirst)]

In [11]:
cntry_df = return_pub_with_first(cntry_df)

In [5]:
author_path ='../../data/dropbox/LaborDivision/Data/rawdata/article_author_info.txt'
author_info = pd.read_csv(author_path, sep=",",on_bad_lines='skip')
author_info.head()

/tmp/ipykernel_457351/1238032764.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  author_info = pd.read_csv(author_path, sep=",",on_bad_lines='skip')


,OST_BK,Seq_No,cluster_id,gender,YFP,npapa
0,107154401,5.0,81178534.0,F,2012.0,11.0
1,108260661,745.0,4745329.0,F,2013.0,746.0
2,105432460,9.0,30161693.0,U,2013.0,1.0
3,40628747,1.0,88125653.0,M,2003.0,111.0
4,50934034,6.0,22288059.0,M,2000.0,258.0


In [12]:
def return_pub_with_author(pub_df, author_df):
    merged_df = pub_df.merge(author_df, on=['OST_BK','Seq_No'], how='left')
    invalid_ids = merged_df[merged_df.cluster_id.isna()].OST_BK.unique()
    valid_df = pub_df[~pub_df.OST_BK.isin(invalid_ids)]
    return valid_df

In [13]:
cntry_df = return_pub_with_author(cntry_df, author_info)

In [15]:
cntry_df[cntry_df.OST_BK==80146396]

,OST_BK,Seq_No,Last_Name,Addr_No,Country
1,80146396,8.0,Evans,2.0,Iraq
9022512,80146396,6.0,Morton,4.0,USA
10498645,80146396,7.0,Hasoon,2.0,Iraq
11317794,80146396,1.0,Donaldson,1.0,USA
17586128,80146396,2.0,Shanovich,1.0,USA
30289917,80146396,4.0,Clark,2.0,Iraq
53361488,80146396,3.0,Shetty,1.0,USA
74096716,80146396,5.0,Aziz,2.0,Iraq
104273014,80146396,5.0,Aziz,3.0,Iraq
112481200,80146396,1.0,Donaldson,2.0,Iraq


In [16]:
def remove_multi_affiliation(df):
    new_df = df.sort_values(by=['OST_BK','Seq_No','Addr_No']
                           ).drop_duplicates(subset=['OST_BK','Seq_No'],keep='first')
    return new_df

In [17]:
cntry_df = remove_multi_affiliation(cntry_df)

In [18]:
cntry_df

,OST_BK,Seq_No,Last_Name,Addr_No,Country
122933287,29768068,1.0,Miron,1.0,Canada
39746120,29768068,2.0,Hall,3.0,Canada
95825479,29768068,5.0,Antel,1.0,Canada
15073160,29768070,1.0,Shen,1.0,Japan
89644398,29768070,2.0,Tanida,1.0,Japan
...,...,...,...,...,...
69044562,108741050,4.0,Perrin,1.0,USA
94898141,108741050,5.0,Arango-Lasprilla,4.0,Spain
93966061,108741055,1.0,Flint,1.0,England
81801811,108741055,2.0,Fearon,1.0,England


In [7]:
fund_path = '../../data/dropbox/LaborDivision/Data/rawdata/article_funding.txt'
fund_df = pd.read_csv(fund_path, sep="|", on_bad_lines='skip')
fund_df.head()

,OST_BK,Agency
0,89364646,Funding Program for Next Generation World-Lead...
1,89364646,JSPS
2,89364646,"Ministry of Education, Culture, Sports, Scienc..."
3,89364646,Japan Society for the Promotion of Science
4,89364646,Japan Society for the Promotion of Science (JSPS)


In [8]:
pub_path = "/u/miaoli/ember_home/dropbox/helicopter-disease/data/rawdata/CVDPilotData/GBD_CVD_2000_2019_AllSexesAges.dta"

In [9]:
pd.read_stata(pub_path)

,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,2,DALYs (Disability-Adjusted Life Years),67,Japan,3,Both,22,All ages,970,Other non-rheumatic valve diseases,1,Number,2000,278.490570,404.885040,173.320862
1,2,DALYs (Disability-Adjusted Life Years),67,Japan,3,Both,22,All ages,970,Other non-rheumatic valve diseases,2,Percent,2000,0.000008,0.000012,0.000005
2,2,DALYs (Disability-Adjusted Life Years),67,Japan,3,Both,22,All ages,970,Other non-rheumatic valve diseases,3,Rate,2000,0.215832,0.313789,0.134325
3,2,DALYs (Disability-Adjusted Life Years),99,Uruguay,3,Both,22,All ages,1023,Other cardiovascular and circulatory diseases,1,Number,2000,5072.336914,5552.016602,4610.193359
4,2,DALYs (Disability-Adjusted Life Years),99,Uruguay,3,Both,22,All ages,1023,Other cardiovascular and circulatory diseases,2,Percent,2000,0.004833,0.005258,0.004445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257035,2,DALYs (Disability-Adjusted Life Years),183,Mauritius,3,Both,22,All ages,942,Myocarditis,2,Percent,2007,0.000116,0.000143,0.000080
257036,2,DALYs (Disability-Adjusted Life Years),183,Mauritius,3,Both,22,All ages,942,Myocarditis,3,Rate,2007,3.445148,4.146368,2.379969
257037,2,DALYs (Disability-Adjusted Life Years),183,Mauritius,3,Both,22,All ages,944,Other cardiomyopathy,1,Number,2007,1082.817871,1210.220337,789.416016
257038,2,DALYs (Disability-Adjusted Life Years),183,Mauritius,3,Both,22,All ages,944,Other cardiomyopathy,2,Percent,2007,0.002898,0.003370,0.002093
